# Book spine detector — YOLOv8-OBB, trained on Colab's GPU

Run cells top to bottom. Runtime -> Change runtime type -> GPU (T4 is fine; A100 if you have Colab Pro).

Get the dataset onto Colab one of two ways (use whichever cell applies, skip the other):
- **A. Roboflow API** (fastest, no manual upload): needs your Roboflow API key and the project's workspace/version.
- **B. Manual upload**: zip `Book Spines test.v6i.yolov8-obb` yourself and upload it in the cell below.

In [ ]:
!pip install -q ultralytics roboflow
import torch
print('CUDA available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime > Change runtime type > GPU')

## Option A — pull the dataset from Roboflow directly

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# from the dataset's Roboflow URL: universe.roboflow.com/<workspace>/<project>/dataset/<version>
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("yolov8-obb")
DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)

## Option B — upload the dataset zip manually instead

In [ ]:
from google.colab import files
import zipfile, glob

uploaded = files.upload()  # pick the zipped 'Book Spines test.v6i.yolov8-obb' folder
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("dataset")
DATA_YAML = glob.glob("dataset/**/data.yaml", recursive=True)[0]
print(DATA_YAML)

## Fix the data.yaml `path:` to the extracted location, then train

`MODEL` picks the size (`yolov8n-obb.pt` .. `yolov8x-obb.pt`); Colab's T4 (16GB) comfortably fits `x` at
batch 16+, unlike the 4GB local card this was prototyped on.

In [ ]:
import yaml, os

d = yaml.safe_load(open(DATA_YAML))
d["path"] = os.path.dirname(DATA_YAML)
yaml.safe_dump(d, open(DATA_YAML, "w"))
print(d)

In [ ]:
from ultralytics import YOLO

MODEL = "yolov8x-obb.pt"   # n / s / m / l / x - x is the "high end" option
EPOCHS = 100                # paper's own best book-spine run used 48; Ultralytics default is 100
IMGSZ = 1024                 # matches the paper's input resolution; drop to 640 if VRAM is tight
BATCH = 16                   # T4 16GB fits this comfortably at imgsz 640; reduce if you raise IMGSZ and OOM

model = YOLO(MODEL)
results = model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
                       project="runs_spine", name="colab_run")

## Validate, then download the trained weights

In [ ]:
metrics = model.val()
print(metrics.box.map50, metrics.box.map)  # mAP50, mAP50-95

In [ ]:
from google.colab import files
files.download("runs_spine/colab_run/weights/best.pt")